# 06 · 배포 전 검증과 기획서 개선

목표: HTTP 성공·내용 기준을 따로 평가하고 배포/중단 결정을 기록한다. 약15분.

**실행:** Python 3.11 공통 .venv 커널에서 위→아래. 다른 Notebook·서버·Docker·모델 설치 없이 독립 실행. 코드는 MOCK 고정 문장·대기만 사용한다. 실제 LLM 품질·처리량 측정이 아니다.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from week10.lab.config import Settings
from week10.lab.experiments import lab_client, fixture_check, compare_releases

## 릴리스 비교

v2는 의도적으로 잘못된 고정 문장을 내보낸다. 실제 모델 성능 비교가 아니다.

In [ ]:
rows = await compare_releases()
rows

## HTTP와 내용 기준

동일 입력4건씩인 합성 fixture다. 표본을 전체 서비스 정확도로 일반화하지 않는다.

In [ ]:
assert [r["http_success"] for r in rows] == [4, 4]
assert [r["fixture_pass"] for r in rows] == [4, 0]
[(r["release"], r["http_success"] / r["requests"], r["fixture_pass"] / r["requests"]) for r in rows]

## 배포 차단 조건

이번 fixture 전부 통과를 데모 gate로 사용한다. 운영 기준은 프로젝트에 맞게 별도 정한다.

In [ ]:
def gate(row):
    return row["http_success"] == row["requests"] and row["fixture_pass"] == row["requests"]


assert gate(rows[0]) and not gate(rows[1])
print({r["release"]: gate(r) for r in rows})

## 이전 버전 재검증

이전 설정 앱을 새로 시작해 확인하는 로컬 시뮬레이션이다. 실제 트래픽 전환은 수행하지 않는다.

In [ ]:
async with lab_client(Settings(release="v1")) as (app, client):
    r = await client.post("/api/v1/assist", json={"message": "신청 절차"})
    assert fixture_check(r.json()["answer"])
    print((await client.get("/release")).json())

## 서로 다른 품질 위험

단순 문자열 규칙의 우회 사례를 만든다. 더 나은 평가에는 검수된 정답·다양한 부정 예시가 필요하다.

In [ ]:
tricky = "D01 담당자 확인이라고 쓰지만 무조건 승인됩니다."
assert not fixture_check(tricky)
print("규칙은 작은 회귀 방지 장치이며 의미 이해 평가가 아니다.")

## 프로젝트 증거의 계층

기획 주장·실행·검증·남은 위험을 연결한다. 새 파일을 자동 저장하거나 기존 작성본을 덮어쓰지 않는다.

In [ ]:
record = {
    "requirement": "신청 절차 안내",
    "release": "v1",
    "contract": "200+schema",
    "content_check": "D01+담당자 확인",
    "failures": ["502", "503", "504"],
    "evidence": "실제 실행 결과 첨부",
    "unverified": "실제 모델·권한·부하·운영배포",
}
record

## 2차 기획서 제출

2026-11-06 금요일 오늘 제출. 미제출자는 늦어도 2026-11-07 토요일까지. 별도 시각·감점 규칙을 추정하지 않는다.

In [ ]:
form = ROOT / "curriculum/2026-2/web/project-portfolio-week10.html"
assert form.exists()
print(form)
print("아주대AI대학원: LMS / 군산대·창원대 등 별도 수강생: 공지된 이메일")

## 프로젝트 적용 기록

예상 → 관찰 → 근거 → 다음 변경을 한 줄씩 기록한다. 실제로 실행하지 않은 조건은 미측정으로 표시한다. 학생 정보나 원문 입력은 공개 저장소에 올리지 않는다.